In [1]:
import random 
import pandas as pd
from haversine import haversine, Unit
import numpy as np

In [2]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from scripts.PHMM.viterbi_algorithm.states import *
from scripts.PHMM.viterbi_algorithm.viterbi import *
from scripts.PHMM.distributions.AIS_RF_distribution import *
import pandas as pd

In [4]:
df = pd.read_parquet("../data/processed/PHMM_sequence_data.parquet")
df

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,0,209112000,0,2024-01-03 13:19:57,NaT,NaT,None,None,begin
1,0,209112000,0,2024-01-03 13:19:58,2024-01-03 13:19:58,NaT,"[48.51548, -126.51201]",None,AIS
2,0,209112000,0,2024-01-03 13:32:50,2024-01-03 13:32:50,NaT,"[48.51033, -126.45776]",None,AIS
3,0,209112000,0,2024-01-03 13:34:50,2024-01-03 13:34:50,NaT,"[48.50968, -126.44914]",None,AIS
4,0,209112000,0,2024-01-03 13:40:59,2024-01-03 13:40:59,2024-01-03 13:41:00,"[48.50753, -126.42323]","[48.499443942360536, -126.4146031322306]",M
...,...,...,...,...,...,...,...,...,...
471750,710,677059800,0,2024-01-08 08:30:39,2024-01-08 08:30:39,NaT,"[19.00953, -68.1877]",None,AIS
471751,710,677059800,0,2024-01-08 08:47:23,2024-01-08 08:47:23,NaT,"[19.02076, -68.22079]",None,AIS
471752,710,677059800,0,2024-01-08 08:50:39,2024-01-08 08:50:39,NaT,"[19.02278, -68.22751]",None,AIS
471753,710,677059800,0,2024-01-08 09:41:00,NaT,2024-01-08 09:41:00,None,"[19.024754315587664, -68.23688644056789]",RF


In [6]:
begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
model = PHMM(states, end)

### __Make test sequence (track 223)__

In [7]:
sequence = df[df['TrackID'] == 223]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None]

In [8]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

In [9]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(431, 'RF'), (586, 'RF'), (598, 'RF')],       
The indexes of RF and/or M of the real path are: [(430, 'M'), (584, 'M'), (595, 'M')]


### __Make test sequence (track 18)__

In [10]:
sequence2 = df[df['TrackID'] == 18]

AIS_sequence2 = [(time, coord) for time, coord in zip(sequence2["AIS_Timestamp"], sequence2["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence2 = [(time, coord) for time, coord in zip(sequence2["RF_Timestamp"], sequence2["RF"]) 
               if pd.notna(time) and coord is not None]

In [11]:
path2, probability2, log_probs2 = model.viterbi(AIS_sequence2, RF_sequence2)

In [12]:
RF_M_indexes_pred_path2 = [(i, state) for i, state in enumerate(path2) if state in {"RF", "M"}]
RF_M_indexes_real_path2 = [(i, state) for i, state in enumerate(sequence2["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path2}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path2}")

The indexes of RF and/or M of predicted path are: [(585, 'RF'), (1000, 'RF'), (1089, 'RF')],       
The indexes of RF and/or M of the real path are: [(584, 'M'), (999, 'M'), (1087, 'M')]


In [13]:
import math

In [ ]:
# 1. Selecteer de eerste 100 unieke TrackIDs
first_200_trackids = df["TrackID"].drop_duplicates().head(200)

# 2. Filter het DataFrame op deze TrackIDs
df_first_200 = df[df["TrackID"].isin(first_200_trackids)]

In [28]:
RF_data = df_first_200[df_first_200["RF"].notna()].copy()
RF_data["RF_data_point_id"] = RF_data.groupby("TrackID").cumcount() + 1

M_data = df_first_200[df_first_200["State"] == "M"].copy()
M_data["distance_km"] = M_data.apply(lambda row: haversine(row["AIS"], row["RF"], unit=Unit.KILOMETERS), axis=1)
radius_km = math.ceil(M_data["distance_km"].max())

viterbi_results = []
speed_data = []

for _, RF_data_point in RF_data.iterrows():
    RF_coord = RF_data_point["RF"]
    RF_time = RF_data_point["RF_Timestamp"]
    
    for AIS_id, AIS_data in df_first_200.groupby("TrackID"):
        AIS_coords = [coord for coord in AIS_data["AIS"] if coord is not None]
        RF_within_radius = any(haversine(RF_coord, coord) <= radius_km for coord in AIS_coords)
        
        marge = pd.Timedelta(minutes=10)
        min_AIS_time = AIS_data["AIS_Timestamp"].min() - marge
        max_AIS_time = AIS_data["AIS_Timestamp"].max() + marge
        RF_within_time = min_AIS_time <= RF_time <= max_AIS_time
        
        if RF_within_radius and RF_within_time:
            AIS_seq = [(time, coord) 
                       for time, coord in zip(AIS_data["AIS_Timestamp"], AIS_data["AIS"]) 
                       if pd.notna(time) and coord is not None]
            RF_seq = [(RF_time, RF_coord)]
            
            path, prob, log_probs = model.viterbi(AIS_seq, RF_seq)
            
            for i in range(len(AIS_seq)-1):
                time_curr_row, coord_curr_row = AIS_seq[i]
                time_next_row, coord_next_row = AIS_seq[i+1]
                
                delta_time_sec = (time_next_row - time_curr_row).total_seconds()
                delta_dist_km = haversine(coord_curr_row, coord_next_row)
                
                if delta_time_sec > 0 and delta_dist_km > 0:
                    speed_kmph = delta_dist_km / (delta_time_sec / 3600)
                    speed_data.append(speed_kmph) 
            
            avg_speed = np.mean(speed_data) if speed_data else 0.0 
            normalized_prob = prob/len(AIS_seq)
            speed_normalized_prob = normalized_prob/avg_speed
            
            viterbi_results.append(
                {
                    "RF_sequence_id": RF_data_point["TrackID"],
                    "RF_data_point_id": RF_data_point["RF_data_point_id"],
                    "AIS_sequence_id": AIS_id,
                    "num_of_AIS_points": len(AIS_seq),
                    "average_speed": avg_speed,
                    "viterbi_prob": prob,
                    "normalized_viterbi_prob": normalized_prob,
                    "speed_normalized_prob": speed_normalized_prob,
                    "match": RF_data_point["TrackID"] == AIS_id,
                    "predicted_path": path
                })      

In [29]:
df_viterbi_results = pd.DataFrame(viterbi_results)
df_viterbi_results

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,average_speed,viterbi_prob,normalized_viterbi_prob,speed_normalized_prob,match,predicted_path
0,0,1,0,885,18.102813,-29.701295,-0.033561,-0.001854,True,"[begin, AIS, AIS, AIS, AIS, RF, AIS, AIS, AIS,..."
1,0,2,0,885,18.102813,-69.199810,-0.078192,-0.004319,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2,0,3,0,885,18.102813,-93.500230,-0.105650,-0.005836,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
3,1,1,1,883,18.316023,-31.781211,-0.035992,-0.001965,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
4,1,2,1,883,18.444064,-51.513695,-0.058339,-0.003163,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
...,...,...,...,...,...,...,...,...,...,...
544,198,4,198,778,15.680227,-83.464978,-0.107281,-0.006842,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
545,199,1,199,981,15.676325,-52.899321,-0.053924,-0.003440,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
546,199,2,199,981,15.672444,-70.672468,-0.072041,-0.004597,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
547,199,3,199,981,15.668584,-84.171282,-0.085802,-0.005476,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [30]:
ids = df_viterbi_results.groupby(["RF_sequence_id", "RF_data_point_id"])["normalized_viterbi_prob"].idxmax()
best_matches = df_viterbi_results.loc[ids].reset_index(drop=True)
best_matches

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,average_speed,viterbi_prob,normalized_viterbi_prob,speed_normalized_prob,match,predicted_path
0,0,1,0,885,18.102813,-29.701295,-0.033561,-0.001854,True,"[begin, AIS, AIS, AIS, AIS, RF, AIS, AIS, AIS,..."
1,0,2,0,885,18.102813,-69.199810,-0.078192,-0.004319,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2,0,3,0,885,18.102813,-93.500230,-0.105650,-0.005836,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
3,1,1,1,883,18.316023,-31.781211,-0.035992,-0.001965,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
4,1,2,1,883,18.444064,-51.513695,-0.058339,-0.003163,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
...,...,...,...,...,...,...,...,...,...,...
490,198,4,119,823,15.672131,-36.408244,-0.044238,-0.002823,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
491,199,1,199,981,15.676325,-52.899321,-0.053924,-0.003440,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
492,199,2,199,981,15.672444,-70.672468,-0.072041,-0.004597,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
493,199,3,199,981,15.668584,-84.171282,-0.085802,-0.005476,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [31]:
best_matches[best_matches["match"] == False]

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,average_speed,viterbi_prob,normalized_viterbi_prob,speed_normalized_prob,match,predicted_path
11,4,1,3,646,16.229023,-66.876788,-0.103524,-0.006379,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
50,18,3,19,668,14.582779,-21.226305,-0.031776,-0.002179,False,"[begin, RF, AIS, AIS, AIS, AIS, AIS, AIS, AIS,..."
54,20,3,21,866,15.831030,-28.731889,-0.033178,-0.002096,False,"[begin, AIS, AIS, RF, AIS, AIS, AIS, AIS, AIS,..."
68,26,1,98,787,15.489713,-46.567287,-0.059171,-0.003820,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
114,46,3,197,349,16.504649,-21.313966,-0.061072,-0.003700,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
147,58,1,57,780,16.110481,-80.886586,-0.103701,-0.006437,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
158,63,1,62,495,15.944279,-55.755911,-0.112638,-0.007064,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
164,66,1,102,534,16.079153,-22.487725,-0.042112,-0.002619,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
165,66,2,102,534,16.059689,-54.125850,-0.101359,-0.006311,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
196,79,1,78,417,16.245726,-44.350542,-0.106356,-0.006547,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [32]:
df_viterbi_results[df_viterbi_results["RF_sequence_id"] == 46]

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,average_speed,viterbi_prob,normalized_viterbi_prob,speed_normalized_prob,match,predicted_path
124,46,1,46,635,16.770440,-43.546360,-0.068577,-0.004089,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
125,46,1,196,568,16.719293,-52.818761,-0.092991,-0.005562,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
126,46,2,46,635,16.649842,-53.328304,-0.083982,-0.005044,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
127,46,3,46,635,16.581364,-65.635408,-0.103363,-0.006234,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
128,46,3,60,729,16.559160,-48.652203,-0.066738,-0.004030,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
129,46,3,197,349,16.504649,-21.313966,-0.061072,-0.003700,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [24]:
stationarity_penalty = 1 / (37.48 + 1e-6)
adjusted_prob = -0.103363 / 37.48
adjusted_prob

-0.00275781750266809

In [23]:
stationarity_penalty = 1 / (1.13 + 1e-6)
adjusted_prob = -0.061072 / 1.13
adjusted_prob

-0.05404601769911505

In [ ]:
df_first_200[(df_first_200["TrackID"] == 46) & ((df_first_200["State"] == "RF")|(df_first_200["State"] == "M"))].head(10)

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
30682,46,232024772,1,2024-01-24 21:41:46,2024-01-24 21:41:46,2024-01-24 21:41:00,"[33.65732, -118.22315]","[33.64856212839301, -118.2289547254344]",M
30802,46,232024772,1,2024-01-25 01:41:00,NaT,2024-01-25 01:41:00,None,"[33.73690009077661, -118.2400597817236]",RF
30958,46,232024772,1,2024-01-25 09:41:00,NaT,2024-01-25 09:41:00,None,"[33.73628020900185, -118.24198403939789]",RF


In [25]:
df_first_200.iloc[30958-2: 30958+2]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
30956,46,232024772,1,2024-01-25 09:36:07,2024-01-25 09:36:07,NaT,"[33.73119, -118.25149]",None,AIS
30957,46,232024772,1,2024-01-25 09:39:07,2024-01-25 09:39:07,NaT,"[33.7312, -118.25148]",None,AIS
30958,46,232024772,1,2024-01-25 09:41:00,NaT,2024-01-25 09:41:00,None,"[33.73628020900185, -118.24198403939789]",RF
30959,46,232024772,1,2024-01-25 09:42:07,2024-01-25 09:42:07,NaT,"[33.73117, -118.25148]",None,AIS


In [26]:
(pd.Timestamp("2024-01-25 09:41:00") - pd.Timestamp("2024-01-25 09:39:07")).total_seconds(), haversine([33.7312, -118.25148],[33.73628020900185, -118.24198403939789])

(113.0, 1.0441253825194825)

In [32]:
AIS_RF_probability(113.0, 1.0441253825194825)

np.float64(0.8638542412603486)

In [24]:
df_first_200[(df_first_200["TrackID"] == 197)]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
126504,197,352002625,3,2024-01-25 03:23:05,NaT,NaT,None,None,begin
126505,197,352002625,3,2024-01-25 03:23:06,2024-01-25 03:23:06,NaT,"[33.71173, -118.20001]",None,AIS
126506,197,352002625,3,2024-01-25 03:26:06,2024-01-25 03:26:06,NaT,"[33.71167, -118.20008]",None,AIS
126507,197,352002625,3,2024-01-25 03:29:06,2024-01-25 03:29:06,NaT,"[33.71158, -118.20017]",None,AIS
126508,197,352002625,3,2024-01-25 03:32:06,2024-01-25 03:32:06,NaT,"[33.7115, -118.20021]",None,AIS
...,...,...,...,...,...,...,...,...,...
126851,197,352002625,3,2024-01-26 03:09:27,2024-01-26 03:09:27,NaT,"[33.72958, -118.25682]",None,AIS
126852,197,352002625,3,2024-01-26 03:12:27,2024-01-26 03:12:27,NaT,"[33.72962, -118.25679]",None,AIS
126853,197,352002625,3,2024-01-26 03:15:27,2024-01-26 03:15:27,NaT,"[33.72958, -118.25677]",None,AIS
126854,197,352002625,3,2024-01-26 03:18:27,2024-01-26 03:18:27,NaT,"[33.7296, -118.2568]",None,AIS


In [29]:
df_first_200.iloc[126577-2: 126577+2]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
126575,197,352002625,3,2024-01-25 09:35:06,2024-01-25 09:35:06,NaT,"[33.71173, -118.19974]",None,AIS
126576,197,352002625,3,2024-01-25 09:38:06,2024-01-25 09:38:06,NaT,"[33.71174, -118.19974]",None,AIS
126577,197,352002625,3,2024-01-25 09:41:06,2024-01-25 09:41:06,2024-01-25 09:41:00,"[33.71179, -118.1997]","[33.70547891161042, -118.19322254441256]",M
126578,197,352002625,3,2024-01-25 09:44:06,2024-01-25 09:44:06,NaT,"[33.71186, -118.19967]",None,AIS


In [30]:
(pd.Timestamp("2024-01-25 09:41:06") - pd.Timestamp("2024-01-25 09:41:00")).total_seconds(), haversine([33.73628020900185, -118.24198403939789],[33.71179, -118.1997])

(6.0, 4.765323776004171)

In [31]:
AIS_RF_probability(6.0, 4.765323776004171)

np.float64(0.7479913456922832)

In [23]:
RF_data[(RF_data["TrackID"] == 197)]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,RF_data_point_id
126577,197,352002625,3,2024-01-25 09:41:06,2024-01-25 09:41:06,2024-01-25 09:41:00,"[33.71179, -118.1997]","[33.70547891161042, -118.19322254441256]",M,1
126826,197,352002625,3,2024-01-26 01:41:00,NaT,2024-01-26 01:41:00,None,"[33.73556434833469, -118.24789054046626]",RF,2


In [34]:
def compute_distance(row):
    """
    Computes the distance between two geographical points using the Haversine formula

    Args:
        row (pd.Series): A row of the DataFrame containing latitude and longitude of two points

    Returns:
        float: Distance in kilometers
    """
    if pd.notnull(row['prev_LAT']) and pd.notnull(row['prev_LON']):
        coord_prev = (row['prev_LAT'], row['prev_LON'])
        coord_curr = (row['LAT'], row['LON'])
        return haversine(coord_prev, coord_curr, unit=Unit.KILOMETERS)
    else:
        return np.nan

def filter_by_speed(df, min_speed = 1, max_speed = 60):
    """
    Filters the AIS data to include only vessels with a speed between min_speed and max_speed

    Args:
        df (pd.DataFrame): AIS data
        min_speed (float): Minimum speed threshold, defaults to 1
        max_speed (float): Maximum speed threshold, defaults to 60

    """
    # Sort the data and reset index for proper sequential processing
    df = df.sort_values(by=["MMSI", "SubTrackID", "BaseDateTime"]).copy()
    
    # Calculate distance between consecutive points
    df["prev_LAT"] = df.groupby(["MMSI", "SubTrackID"])["LAT"].shift(1)
    df["prev_LON"] = df.groupby(["MMSI", "SubTrackID"])["LON"].shift(1)
    df['distance_km'] = df.apply(compute_distance, axis=1)
    
    # Convert time difference to seconds
    df["TimeDiff"] = df["TimeDiff"].dt.total_seconds()
    
    # Calculate speed
    df["speed_kmph"] = 0.0
    valid_rows = (df["TimeDiff"] > 0) & (df["distance_km"] > 0)
    df.loc[valid_rows, "speed_kmph"] = (df.loc[valid_rows, 'distance_km'] / df.loc[valid_rows, 'TimeDiff']) * 3600  
        
    # Calculate average speed for each vessel
    df_avg_speeds = df.groupby(["MMSI", 'SubTrackID'])['speed_kmph'].mean().reset_index()
    
    # Filter vessels based on speed
    valid_tracks = df_avg_speeds[(df_avg_speeds['speed_kmph'] >= min_speed) & (df_avg_speeds['speed_kmph'] <= max_speed)][["MMSI", 'SubTrackID']]
    
    # Merge to get original data
    df_filterd = df.merge(valid_tracks, on=["MMSI", 'SubTrackID'], how='inner')
    
    return df_avg_speeds, df_filterd

In [35]:
df = pd.read_parquet("../data/processed/segmented_tracks.parquet")
df_avg_speeds, df_filterd = filter_by_speed(df)

In [38]:
df_avg_speeds[(df_avg_speeds["MMSI"] == 209112000) & (df_avg_speeds["SubTrackID"] == 0)]

,MMSI,SubTrackID,speed_kmph
0,209112000,0,18.537499


In [ ]:
pd.read_parquet("../data/processed/filtered_AIS_by_speed.parquet")[["MMSI", "SubTrackID"]].drop_duplicates()


,MMSI,SubTrackID
0,209112000,0
983,209112000,20
1964,209112000,21
2050,209444000,0
2974,209444000,1
...,...,...
562112,636023489,5
562634,636023537,0
563323,636092792,0
563382,636092852,0


In [ ]:
pd.read_parquet("../data/processed/modified_AIS_data.parquet")[["MMSI", "SubTrackID"]].drop_duplicates()


,MMSI,SubTrackID
0,209112000,0
885,209112000,20
1768,209112000,21
1827,209444000,0
2516,209444000,1
...,...,...
470057,636023489,5
470461,636023537,0
471012,636092792,0
471065,636092852,0


In [ ]:
pd.read_pickle("../data/processed/train_data/train_set.pkl")[["MMSI", "SubTrackID"]].drop_duplicates()


,MMSI,SubTrackID
0,209112000,0
888,209112000,20
1774,209444000,1
2422,209444000,2
2766,210614000,2
...,...,...
375225,636023489,1
375588,636023489,2
376108,636023537,0
376662,636092852,0


In [ ]:
pd.read_parquet("../data/processed/PHMM_sequence_data.parquet")[["MMSI", "SubTrackID"]].drop_duplicates()


,MMSI,SubTrackID
0,209112000,0
888,209112000,20
1774,209444000,0
2467,209444000,1
3115,209444000,2
...,...,...
469415,636023489,4
469957,636023489,5
470365,636023537,0
470919,636092852,0


In [ ]:
pd.read_pickle("../data/processed/test_data/test_set.pkl")[["MMSI", "SubTrackID"]].drop_duplicates()


,MMSI,SubTrackID
0,209444000,0
693,212273000,0
1448,212589000,0
2065,229752000,1
3044,229752000,6
...,...,...
91240,636018220,4
91681,636019582,6
92269,636022027,0
93307,636023489,4


In [ ]:
568 + 143

711

In [ ]:
s[(s['speed_kmph'] < 1) | (s['speed_kmph'] > 60)]


,MMSI,SubTrackID,speed_kmph
1,209112000,1,0.171137
2,209112000,2,0.239521
3,209112000,3,0.256462
4,209112000,4,0.160448
5,209112000,5,0.257146
...,...,...,...
1495,636023452,0,0.043615
1496,636023452,1,0.040677
1499,636023489,0,0.214687
1502,636023489,3,0.041296


In [ ]:
n[(n["MMSI"] == 244170000) & (n["SubTrackID"] == 1)]["speed_kmph"].sum()/len(n[(n["MMSI"] == 244170000) & (n["SubTrackID"] == 1)]["speed_kmph"])

np.float64(11.167934505139044)

In [ ]:
df_avg_speed = n.groupby(["MMSI", "SubTrackID"])["speed_kmph"].mean().reset_index()
df_avg_speed[(df_avg_speed["MMSI"] == 244170000)]

,MMSI,SubTrackID,speed_kmph
138,244170000,0,14.612762
139,244170000,1,11.167935
140,244170000,2,17.355682
141,244170000,3,20.300327


In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from scripts.PHMM.viterbi_algorithm.states import *
from scripts.PHMM.viterbi_algorithm.viterbi import *
from scripts.PHMM.distributions.AIS_RF_distribution import *
import pandas as pd

### __Get testing data__

In [ ]:
test_set = pd.read_pickle("../data/processed/test_data/test_set.pkl")

### __Initialize states and model__

In [ ]:
begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
model = PHMM(states, end)

In [ ]:
test_set["TrackID"].unique()

array([  2,   6,  10,  30,  31,  39,  41,  44,  49,  54,  55,  56,  60,
        63,  65,  69,  70,  72,  76,  77,  78,  81,  84,  86,  90,  97,
       101, 108, 109, 118, 120, 131, 132, 133, 135, 136, 145, 148, 155,
       158, 164, 165, 174, 181, 192, 196, 199, 204, 208, 209, 210, 213,
       215, 220, 223, 227, 235, 239, 247, 256, 257, 264, 265, 284, 289,
       291, 292, 300, 302, 311, 314, 318, 324, 326, 328, 331, 332, 333,
       336, 340, 346, 354, 356, 363, 377, 381, 394, 396, 404, 405, 408,
       409, 420, 425, 434, 436, 437, 439, 444, 451, 465, 490, 494, 506,
       512, 515, 516, 518, 523, 526, 527, 530, 535, 544, 545, 554, 559,
       567, 580, 583, 584, 588, 589, 596, 598, 606, 609, 611, 615, 619,
       628, 631, 643, 644, 651, 659, 660, 665, 670, 677, 697, 706, 707])

In [ ]:
sequence = test_set[test_set['TrackID'] == 318]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None]

### __Run Viterbi model of first sequence__

In [ ]:
path, probability = model.viterbi(AIS_sequence, RF_sequence)

In [ ]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(11, 'M'), (293, 'RF'), (503, 'M'), (675, 'RF')],       
The indexes of RF and/or M of the real path are: [(11, 'M'), (293, 'M'), (503, 'M'), (674, 'RF')]


In [ ]:
sequence = test_set[test_set['TrackID'] == 10]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None]

### __Run Viterbi model of first sequence__

In [ ]:
path, probability = model.viterbi(AIS_sequence, RF_sequence)

In [ ]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(7, 'M'), (454, 'M'), (492, 'RF')],       
The indexes of RF and/or M of the real path are: [(7, 'M'), (454, 'M'), (492, 'M')]


In [ ]:
# Step 1: Group and aggregate unique states per sequence
state_per_sequence = test_set.groupby(["TrackID"])["State"].unique().reset_index()
# Step 2: Filter rows that do NOT contain "M"
no_match_sequences = state_per_sequence[~state_per_sequence["State"].apply(lambda states: "M" in states)]
no_match_sequences

,TrackID,State
5,39,"[begin, AIS, RF, end]"
7,44,"[begin, AIS, RF, end]"
13,63,"[begin, RF, AIS, end]"
23,86,"[begin, RF, AIS, end]"
47,204,"[begin, AIS, RF, end]"
49,209,"[begin, AIS, RF, end]"
55,227,"[begin, AIS, RF, end]"
61,264,"[begin, AIS, RF, end]"
73,326,"[begin, AIS, RF, end]"
74,328,"[begin, AIS, RF, end]"


In [ ]:
sequence = test_set[test_set['TrackID'] == 264] # 39, 544, 264, 204

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None]

In [ ]:
RF_sequence

[(Timestamp('2024-01-22 09:41:00'), array([ 36.9400483 , -74.58195162])),
 (Timestamp('2024-01-22 21:41:00'), array([ 37.17774224, -76.11216519])),
 (Timestamp('2024-01-23 01:41:00'), array([ 37.19148019, -76.11684882]))]

### __Run Viterbi model of first sequence__

In [ ]:
path, probability = model.viterbi(AIS_sequence, RF_sequence)

In [ ]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(19, 'RF'), (459, 'RF'), (540, 'RF')],       
The indexes of RF and/or M of the real path are: [(19, 'RF'), (459, 'RF'), (540, 'RF')]


In [ ]:
from haversine import haversine, Unit
import math 

In [ ]:
# Ontleedt dit helemaal en check of het werkt 
RF_data = test_set[test_set["RF"].notna()].copy()
RF_data["RF_data_point_id"] = RF_data.groupby("TrackID").cumcount() + 1

M_data = test_set[test_set["State"] == "M"].copy()
M_data["distance_km"] = M_data.apply(lambda row: haversine(row["AIS"], row["RF"], unit=Unit.KILOMETERS), axis=1)
radius_km = math.ceil(M_data["distance_km"].max())

viterbi_results = []

for _, RF_data_point in RF_data.iterrows():
    RF_coord = RF_data_point["RF"]
    RF_time = RF_data_point["RF_Timestamp"]
    
    for AIS_id, AIS_data in test_set.groupby("TrackID"):
        AIS_coords = [coord for coord in AIS_data["AIS"] if coord is not None]
        RF_within_radius = any(haversine(RF_coord, coord) <= radius_km for coord in AIS_coords)
        
        marge = pd.Timedelta(minutes=10)
        min_AIS_time = AIS_data["AIS_Timestamp"].min() - marge
        max_AIS_time = AIS_data["AIS_Timestamp"].max() + marge
        RF_within_time = min_AIS_time <= RF_time <= max_AIS_time
        
        if RF_within_radius and RF_within_time:
            AIS_seq = [(time, coord) 
                       for time, coord in zip(AIS_data["AIS_Timestamp"], AIS_data["AIS"]) 
                       if pd.notna(time) and coord is not None]
            RF_seq = [(RF_time, RF_coord)]
            
            path, prob = model.viterbi(AIS_seq, RF_seq)
            
            viterbi_results.append(
                {
                    "RF_sequence_id": RF_data_point["TrackID"],
                    "RF_data_point_id": RF_data_point["RF_data_point_id"],
                    "AIS_sequence_id": AIS_id,
                    "num_of_AIS_points": len(AIS_seq),
                    "viterbi_prob": prob,
                    "match": RF_data_point["TrackID"] == AIS_id,
                    "predicted_path": path
                })
            

In [ ]:
df_viterbi_results = pd.DataFrame(viterbi_results)
df_viterbi_results

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,match,predicted_path
0,2,1,2,691,-63.753106,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
1,2,2,2,691,-43.216176,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2,2,3,2,691,-58.166710,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
3,6,1,6,753,-69.575019,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
4,6,2,6,753,-56.551703,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
...,...,...,...,...,...,...,...
400,706,2,706,538,-48.079891,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
401,706,3,490,560,-59.693685,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
402,706,3,706,538,-71.483759,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
403,707,1,706,538,-57.569953,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [ ]:
ids = df_viterbi_results.groupby(["RF_sequence_id", "RF_data_point_id"])["viterbi_prob"].idxmax()
best_matches = df_viterbi_results.loc[ids].reset_index(drop=True)
best_matches

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,match,predicted_path
0,2,1,2,691,-63.753106,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
1,2,2,2,691,-43.216176,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2,2,3,2,691,-58.166710,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
3,6,1,6,753,-69.575019,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
4,6,2,6,753,-56.551703,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
...,...,...,...,...,...,...,...
354,706,1,706,538,-47.037281,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
355,706,2,706,538,-48.079891,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
356,706,3,490,560,-59.693685,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
357,707,1,706,538,-57.569953,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [ ]:
best_matches[best_matches["match"] == False]

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,match,predicted_path
41,70,3,49,515,-31.989911,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
42,70,4,518,495,-40.331282,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
64,90,2,598,705,-42.952695,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
70,109,2,420,661,-51.529259,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
72,118,1,512,534,-68.746065,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
87,136,1,97,686,-51.640252,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
97,164,1,291,1022,-63.427858,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
98,164,2,291,1022,-74.961829,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
99,164,3,291,1022,-61.872262,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
108,196,3,311,837,-52.809828,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [ ]:
test_set[(test_set["TrackID"] == 291) & (test_set["State"] == "M")].index

Index([42172, 42587], dtype='int64')

In [ ]:
test_set.iloc[42172-5:42172+5]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
42167,291,366904890,2,2024-01-03 13:36:01,2024-01-03 13:36:01,NaT,"[43.62623, -82.46805]",None,AIS
42168,291,366904890,2,2024-01-03 13:37:10,2024-01-03 13:37:10,NaT,"[43.62922, -82.46859]",None,AIS
42169,291,366904890,2,2024-01-03 13:38:21,2024-01-03 13:38:21,NaT,"[43.63226, -82.46914]",None,AIS
42170,291,366904890,2,2024-01-03 13:39:30,2024-01-03 13:39:30,NaT,"[43.63522, -82.46966]",None,AIS
42171,291,366904890,2,2024-01-03 13:40:31,2024-01-03 13:40:31,NaT,"[43.63784, -82.4702]",None,AIS
42172,291,366904890,2,2024-01-03 13:41:41,2024-01-03 13:41:41,2024-01-03 13:41:00,"[43.6409, -82.47073]","[43.650327037186216, -82.47302729619881]",M
42173,291,366904890,2,2024-01-03 13:42:50,2024-01-03 13:42:50,NaT,"[43.64387, -82.47128]",None,AIS
42174,291,366904890,2,2024-01-03 13:44:01,2024-01-03 13:44:01,NaT,"[43.64688, -82.47184]",None,AIS
42175,291,366904890,2,2024-01-03 13:45:10,2024-01-03 13:45:10,NaT,"[43.64986, -82.47236]",None,AIS
42176,291,366904890,2,2024-01-03 13:46:19,2024-01-03 13:46:19,NaT,"[43.65288, -82.47291]",None,AIS


In [ ]:
k = pd.read_parquet("../data/processed/modified_AIS_data.parquet")
k

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,...,Width,Draft,Cargo,TransceiverClass,TimeDiff,SubTrackID,prev_LAT,prev_LON,distance_km,speed_kmph
0,209112000,2024-01-03 13:19:58,48.51548,-126.51201,9.8,98.7,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,NaN,0,NaN,NaN,NaN,0.000000
1,209112000,2024-01-03 13:32:50,48.51033,-126.45776,10.3,99.7,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,772.0,0,48.51548,-126.51201,4.036950,18.825155
2,209112000,2024-01-03 13:34:50,48.50968,-126.44914,10.5,91.4,104.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,120.0,0,48.51033,-126.45776,0.639097,19.172914
3,209112000,2024-01-03 13:40:59,48.50753,-126.42323,10.1,96.0,107.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,369.0,0,48.50968,-126.44914,1.923641,18.767225
4,209112000,2024-01-03 13:42:38,48.50664,-126.41652,9.8,107.4,107.0,STALO,IMO9758404,5BQM4,...,32.0,7.5,70.0,A,99.0,0,48.50753,-126.42323,0.504133,18.332117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
471890,677059800,2024-01-08 08:26:19,19.00699,-68.17823,8.2,293.7,294.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,126.0,0,19.00504,-68.17374,0.519466,14.841884
471891,677059800,2024-01-08 08:28:43,19.00889,-68.18343,7.7,280.4,270.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,144.0,0,19.00699,-68.17823,0.586090,14.652246
471892,677059800,2024-01-08 08:30:39,19.00953,-68.18770,7.8,293.4,309.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,116.0,0,19.00889,-68.18343,0.454516,14.105669
471893,677059800,2024-01-08 08:47:23,19.02076,-68.22079,7.8,253.3,265.0,ZULU AJ,IMO8315449,5IM698,...,10.0,3.0,70.0,A,1004.0,0,19.00953,-68.18770,3.696002,13.252595


In [ ]:
# Probeer hier is gewoon rf sequence 164 met ais sequence 164
# En rf sequence 164 en ais sequence 291

In [ ]:
test_set = pd.read_pickle("../data/processed/test_data/test_set.pkl")

In [ ]:
sequence = test_set[test_set['TrackID'] == 164]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None][0:1]

In [ ]:
RF_sequence

[(Timestamp('2024-01-03 01:41:00'), array([ 42.96610909, -82.4186289 ]))]

In [ ]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

In [ ]:
df = test_set[test_set["TrackID"] == 164]
match_rows = df[df["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df = df[(df["State"] != "M") & (df["State"] != "RF")]
df = pd.concat([df, ais_rows, rf_rows.iloc[0:1]], ignore_index=True)
df = df.sort_values(by=["Timestamp"]).reset_index(drop=True) 

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged = df.copy().reset_index(drop=True)
df_merged["BestPathState"] = df_path["State"]
df_merged["LogProb"] = df_path["LogProb"]
df_merged["Probability"] = np.exp(df_path["LogProb"])

df_merged["prev_Timestamp"] = df_merged["Timestamp"].shift(1)
df_merged["delta_time_sec"] = (df_merged["Timestamp"] - df_merged["prev_Timestamp"]).dt.total_seconds()

df_merged["coord"] = df_merged["AIS"].combine_first(df_merged["RF"])
df_merged["prev_coord"] = df_merged["coord"].shift(1)
df_merged["distance_km"] = None  

for idx, row in df_merged.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged.at[idx, "distance_km"] = dist
df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km
0,164,316009457,2,2024-01-03 00:01:08,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None
1,164,316009457,2,2024-01-03 00:01:09,2024-01-03 00:01:09,NaT,"[43.26082, -82.44656]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:01:08,1.0,"[43.26082, -82.44656]",None,None
2,164,316009457,2,2024-01-03 00:02:59,2024-01-03 00:02:59,NaT,"[43.25506, -82.44536]",None,AIS,AIS,-1.282521,2.773373e-01,2024-01-03 00:01:09,110.0,"[43.25506, -82.44536]","[43.26082, -82.44656]",0.647814
3,164,316009457,2,2024-01-03 00:04:09,2024-01-03 00:04:09,NaT,"[43.2514, -82.44465]",None,AIS,AIS,-1.469395,2.300647e-01,2024-01-03 00:02:59,70.0,"[43.2514, -82.44465]","[43.25506, -82.44536]",0.411016
4,164,316009457,2,2024-01-03 00:05:20,2024-01-03 00:05:20,NaT,"[43.24775, -82.4439]",None,AIS,AIS,-1.659330,1.902665e-01,2024-01-03 00:04:09,71.0,"[43.24775, -82.4439]","[43.2514, -82.44465]",0.410383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
917,164,316009457,2,2024-01-03 23:56:49,2024-01-03 23:56:49,NaT,"[42.26315, -83.10541]",None,AIS,AIS,-78.873356,5.568482e-35,2024-01-03 23:55:40,69.0,"[42.26315, -83.10541]","[42.26315, -83.10539]",0.001646
918,164,316009457,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[42.26315, -83.10541]",None,AIS,AIS,-78.924649,5.290058e-35,2024-01-03 23:56:49,71.0,"[42.26315, -83.10541]","[42.26315, -83.10541]",0.0
919,164,316009457,2,2024-01-03 23:59:01,2024-01-03 23:59:01,NaT,"[42.26315, -83.1054]",None,AIS,AIS,-78.975942,5.025555e-35,2024-01-03 23:58:00,61.0,"[42.26315, -83.1054]","[42.26315, -83.10541]",0.000823
920,164,316009457,2,2024-01-04 00:00:00,2024-01-04 00:00:00,NaT,"[42.26315, -83.1054]",None,AIS,AIS,-79.027236,4.774277e-35,2024-01-03 23:59:01,59.0,"[42.26315, -83.1054]","[42.26315, -83.1054]",0.0


In [ ]:
df_merged.iloc[159-1:159+5]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km
158,164,316009457,2,2024-01-03 03:03:30,2024-01-03 03:03:30,NaT,"[42.76182, -82.46745]",None,AIS,AIS,-39.941745,4.503192e-18,2024-01-03 03:02:20,70.0,"[42.76182, -82.46745]","[42.76517, -82.46694]",0.374823
159,164,316009457,2,2024-01-03 03:04:39,2024-01-03 03:04:39,NaT,"[42.75882, -82.46882]",None,AIS,AIS,-39.993039,4.278032e-18,2024-01-03 03:03:30,69.0,"[42.75882, -82.46882]","[42.76182, -82.46745]",0.351836
160,164,316009457,2,2024-01-03 05:29:40,2024-01-03 05:29:40,NaT,"[42.45249, -82.77025]",None,AIS,AIS,-40.044332,4.064130e-18,2024-01-03 03:04:39,8701.0,"[42.45249, -82.77025]","[42.75882, -82.46882]",42.057645
161,164,316009457,2,2024-01-03 05:30:49,2024-01-03 05:30:49,NaT,"[42.45025, -82.77368]",None,AIS,AIS,-40.095625,3.860924e-18,2024-01-03 05:29:40,69.0,"[42.45025, -82.77368]","[42.45249, -82.77025]",0.375811
162,164,316009457,2,2024-01-03 05:32:00,2024-01-03 05:32:00,NaT,"[42.44791, -82.7772]",None,AIS,AIS,-40.146918,3.667878e-18,2024-01-03 05:30:49,71.0,"[42.44791, -82.7772]","[42.45025, -82.77368]",0.388733
163,164,316009457,2,2024-01-03 05:33:09,2024-01-03 05:33:09,NaT,"[42.44567, -82.78062]",None,AIS,AIS,-40.198212,3.484484e-18,2024-01-03 05:32:00,69.0,"[42.44567, -82.78062]","[42.44791, -82.7772]",0.375213


In [ ]:
AIS_RF_probability(8701.0, 42.057645)

np.float64(0.060439363363303)

In [ ]:
print(f"time: {df_merged['delta_time_sec'].mean()}, distance: {df_merged['distance_km'].mean()}")

time: 93.73832790445168, distance: 0.15264899839240148


In [ ]:
AIS_RF_probability(93.73832790445168,  0.15264899839240148)

np.float64(0.9746037343047199)

In [ ]:
df_merged['distance_km'].max()

42.05764469177223

In [ ]:
sequence = test_set[test_set['TrackID'] == 291]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

In [ ]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

In [ ]:
df = test_set[test_set["TrackID"] == 164]
match_rows = df[df["State"] == "M"]

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df2 = test_set[test_set["TrackID"] == 291]

match_rows = df2[df2["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

df2 = df2[(df2["State"] != "M") & (df2["State"] != "RF")]
df2 = pd.concat([df2, ais_rows, rf_rows[0:1]], ignore_index=True)
df2 = df2.sort_values(by=["Timestamp"]).reset_index(drop=True) 

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged = df2.copy().reset_index(drop=True)
df_merged["BestPathState"] = df_path["State"]
df_merged["LogProb"] = df_path["LogProb"]
df_merged["Probability"] = np.exp(df_path["LogProb"])

df_merged["prev_Timestamp"] = df_merged["Timestamp"].shift(1)
df_merged["delta_time_sec"] = (df_merged["Timestamp"] - df_merged["prev_Timestamp"]).dt.total_seconds()

df_merged["coord"] = df_merged["AIS"].combine_first(df_merged["RF"])
df_merged["prev_coord"] = df_merged["coord"].shift(1)
df_merged["distance_km"] = None  

for idx, row in df_merged.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged.at[idx, "distance_km"] = dist
df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km
0,291,366904890,2,2024-01-03 00:00:03,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None
1,291,366904890,2,2024-01-03 00:00:04,2024-01-03 00:00:04,NaT,"[42.27507, -83.10986]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:00:03,1.0,"[42.27507, -83.10986]",None,None
2,291,366904890,2,2024-01-03 00:01:06,2024-01-03 00:01:06,NaT,"[42.27466, -83.10859]",None,AIS,AIS,-1.202030,3.005834e-01,2024-01-03 00:00:04,62.0,"[42.27466, -83.10859]","[42.27507, -83.10986]",0.114003
3,291,366904890,2,2024-01-03 00:02:54,2024-01-03 00:02:54,NaT,"[42.27421, -83.10593]",None,AIS,AIS,-1.306942,2.706463e-01,2024-01-03 00:01:06,108.0,"[42.27421, -83.10593]","[42.27466, -83.10859]",0.224503
4,291,366904890,2,2024-01-03 00:03:57,2024-01-03 00:03:57,NaT,"[42.27449, -83.10459]",None,AIS,AIS,-1.412666,2.434932e-01,2024-01-03 00:02:54,63.0,"[42.27449, -83.10459]","[42.27421, -83.10593]",0.114563
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,291,366904890,2,2024-01-03 23:55:41,2024-01-03 23:55:41,NaT,"[45.07417, -83.10874]",None,AIS,AIS,-60.278245,6.629650e-27,2024-01-03 23:54:40,61.0,"[45.07417, -83.10874]","[45.07101, -83.10774]",0.360044
1021,291,366904890,2,2024-01-03 23:56:51,2024-01-03 23:56:51,NaT,"[45.0778, -83.10986]",None,AIS,AIS,-60.329539,6.298168e-27,2024-01-03 23:55:41,70.0,"[45.0778, -83.10986]","[45.07417, -83.10874]",0.413108
1022,291,366904890,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[45.08135, -83.11099]",None,AIS,AIS,-60.380832,5.983259e-27,2024-01-03 23:56:51,69.0,"[45.08135, -83.11099]","[45.0778, -83.10986]",0.404591
1023,291,366904890,2,2024-01-03 23:59:10,2024-01-03 23:59:10,NaT,"[45.08501, -83.11218]",None,AIS,AIS,-60.432125,5.684096e-27,2024-01-03 23:58:00,70.0,"[45.08501, -83.11218]","[45.08135, -83.11099]",0.417561


In [ ]:
print(f"time: {df_merged['delta_time_sec'].mean()}, distance: {df_merged['distance_km'].mean()}")

time: 84.32421875, distance: 0.5249289700921912


In [ ]:
df_merged['distance_km'].max()

91.83989935394888

In [ ]:
AIS_RF_probability(84.32421875,  0.5249289700921912)

np.float64(0.9714601896450765)

In [ ]:
3.333333e-01

0.3333333

In [ ]:
df_merged[df_merged["State"] == "RF"]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb
54,164,316009457,2,2024-01-03 01:41:00,NaT,2024-01-03 01:41:00,None,"[42.966109086623185, -82.41862890079418]",RF,RF,-10.728923


In [ ]:
probability

np.float64(-63.427857611455835)

In [ ]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(54, 'RF')],       
The indexes of RF and/or M of the real path are: [(54, 'RF'), (488, 'M'), (903, 'M')]


In [ ]:
# transitie prob tussen gap
test_set[(test_set["TrackID"] == 164) & (test_set["AIS_Timestamp"] == "2024-01-03 03:04:39")].index

Index([24053], dtype='int64')

In [ ]:
test_set.iloc[24053: 24053+2]["AIS_Timestamp"]

24053   2024-01-03 03:04:39
24054   2024-01-03 05:29:40
Name: AIS_Timestamp, dtype: datetime64[ns]

In [ ]:
time_diff = test_set.iloc[24053: 24053+2]["AIS_Timestamp"].iloc[1] - test_set.iloc[24053: 24053+2]["AIS_Timestamp"].iloc[0]
dist_diff = haversine(test_set.iloc[24053: 24053+2]["AIS"].iloc[0], test_set.iloc[24053: 24053+2]["AIS"].iloc[1], unit=Unit.KILOMETERS)
time_diff, dist_diff
AIS_RF_probability(time_diff.total_seconds(), dist_diff)

np.float64(0.06043936364273937)

In [ ]:
test_set[(test_set["TrackID"] == 291) & (test_set["AIS_Timestamp"] == "2024-01-03 11:20:50")].index

Index([42164], dtype='int64')

In [ ]:
test_set.iloc[42164: 42164+2]["AIS_Timestamp"]

42164   2024-01-03 11:20:50
42165   2024-01-03 13:33:40
Name: AIS_Timestamp, dtype: datetime64[ns]

In [ ]:
time_diff = test_set.iloc[42164: 42164+2]["AIS_Timestamp"].iloc[1] - test_set.iloc[42164: 42164+2]["AIS_Timestamp"].iloc[0]
dist_diff = haversine(test_set.iloc[42164: 42164+2]["AIS"].iloc[0], test_set.iloc[42164: 42164+2]["AIS"].iloc[1], unit=Unit.KILOMETERS)
time_diff, dist_diff
AIS_RF_probability(time_diff.total_seconds(), dist_diff)

np.float64(0.07663923541561497)

In [ ]:
test_set[(test_set["TrackID"] == 164) & (test_set["AIS_Timestamp"] == "2024-01-03 01:40:11")].index

Index([23980], dtype='int64')

In [ ]:
test_set.iloc[23980: 23980+2]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
23980,164,316009457,2,2024-01-03 01:40:11,2024-01-03 01:40:11,NaT,"[42.97816, -82.41604]",None,AIS
23981,164,316009457,2,2024-01-03 01:41:21,2024-01-03 01:41:21,2024-01-03 01:41:00,"[42.97494, -82.41562]","[42.966109086623185, -82.41862890079418]",M


In [ ]:
time_diff = test_set.iloc[23980: 23980+2]["AIS_Timestamp"].iloc[1] - test_set.iloc[23980: 23980+2]["AIS_Timestamp"].iloc[0]
dist_diff = haversine(test_set.iloc[23980: 23980+2]["AIS"].iloc[0], test_set.iloc[23980: 23980+2]["AIS"].iloc[1], unit=Unit.KILOMETERS)
time_diff, dist_diff
AIS_RF_probability(time_diff.total_seconds(), dist_diff)

np.float64(0.9773649820831369)

In [ ]:
test_set[(test_set["TrackID"] == 291) & (test_set["AIS_Timestamp"] == "2024-01-03 01:39:03")].index

Index([41737], dtype='int64')

In [ ]:
test_set.iloc[41737]

TrackID                            291
MMSI                         366904890
SubTrackID                           2
Timestamp          2024-01-03 01:39:03
AIS_Timestamp      2024-01-03 01:39:03
RF_Timestamp                       NaT
AIS              [42.30321, -83.08833]
RF                                None
State                              AIS
Name: 41737, dtype: object

In [ ]:
test_set.iloc[23980+1]

TrackID                                               164
MMSI                                            316009457
SubTrackID                                              2
Timestamp                             2024-01-03 01:41:21
AIS_Timestamp                         2024-01-03 01:41:21
RF_Timestamp                          2024-01-03 01:41:00
AIS                                 [42.97494, -82.41562]
RF               [42.966109086623185, -82.41862890079418]
State                                                   M
Name: 23981, dtype: object

In [ ]:
time_diff = test_set.iloc[23980+1]["RF_Timestamp"] - test_set.iloc[41737]["AIS_Timestamp"]
dist_diff = haversine(test_set.iloc[23980+1]["RF"], test_set.iloc[41737]["AIS"], unit=Unit.KILOMETERS)
time_diff, dist_diff
AIS_RF_probability(time_diff.total_seconds(), dist_diff)

np.float64(0.2449139167680545)

# IK BEN HIER NU BEZIG

RF: 164 en AIS: 164

In [ ]:
begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
model = PHMM(states, end)

In [11]:
test_set = pd.read_parquet("../data/processed/PHMM_sequence_data.parquet")
test_set

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,0,209112000,0,2024-01-03 13:19:57,NaT,NaT,None,None,begin
1,0,209112000,0,2024-01-03 13:19:58,2024-01-03 13:19:58,NaT,"[48.51548, -126.51201]",None,AIS
2,0,209112000,0,2024-01-03 13:32:50,2024-01-03 13:32:50,NaT,"[48.51033, -126.45776]",None,AIS
3,0,209112000,0,2024-01-03 13:34:50,2024-01-03 13:34:50,NaT,"[48.50968, -126.44914]",None,AIS
4,0,209112000,0,2024-01-03 13:40:59,2024-01-03 13:40:59,2024-01-03 13:41:00,"[48.50753, -126.42323]","[48.499443942360536, -126.4146031322306]",M
...,...,...,...,...,...,...,...,...,...
471750,710,677059800,0,2024-01-08 08:30:39,2024-01-08 08:30:39,NaT,"[19.00953, -68.1877]",None,AIS
471751,710,677059800,0,2024-01-08 08:47:23,2024-01-08 08:47:23,NaT,"[19.02076, -68.22079]",None,AIS
471752,710,677059800,0,2024-01-08 08:50:39,2024-01-08 08:50:39,NaT,"[19.02278, -68.22751]",None,AIS
471753,710,677059800,0,2024-01-08 09:41:00,NaT,2024-01-08 09:41:00,None,"[19.024754315587664, -68.23688644056789]",RF


In [12]:
df = test_set[test_set["TrackID"] == 164]
df.head(10)

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
103105,164,316009457,2,2024-01-03 00:01:08,NaT,NaT,None,None,begin
103106,164,316009457,2,2024-01-03 00:01:09,2024-01-03 00:01:09,NaT,"[43.26082, -82.44656]",None,AIS
103107,164,316009457,2,2024-01-03 00:02:59,2024-01-03 00:02:59,NaT,"[43.25506, -82.44536]",None,AIS
103108,164,316009457,2,2024-01-03 00:04:09,2024-01-03 00:04:09,NaT,"[43.2514, -82.44465]",None,AIS
103109,164,316009457,2,2024-01-03 00:05:20,2024-01-03 00:05:20,NaT,"[43.24775, -82.4439]",None,AIS
103110,164,316009457,2,2024-01-03 00:06:31,2024-01-03 00:06:31,NaT,"[43.24402, -82.44318]",None,AIS
103111,164,316009457,2,2024-01-03 00:07:40,2024-01-03 00:07:40,NaT,"[43.24045, -82.44245]",None,AIS
103112,164,316009457,2,2024-01-03 00:08:50,2024-01-03 00:08:50,NaT,"[43.23681, -82.44172]",None,AIS
103113,164,316009457,2,2024-01-03 00:10:01,2024-01-03 00:10:01,NaT,"[43.23305, -82.44096]",None,AIS
103114,164,316009457,2,2024-01-03 00:11:11,2024-01-03 00:11:11,NaT,"[43.22944, -82.44028]",None,AIS


In [13]:
match_rows = df[df["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df = df[(df["State"] != "M") & (df["State"] != "RF")]
df = pd.concat([df, ais_rows, rf_rows.iloc[0:1]], ignore_index=True)
df = df.sort_values(by=["Timestamp"]).reset_index(drop=True) 

len(df)

922

In [14]:
df["prev_state"] = df["State"].shift(1)
df["transition"] = df["prev_state"] + "→" + df["State"]
df["trans_prob"] = np.nan

In [15]:
import numpy as np

# Initieer kolommen
df["prev_log_prob"] = np.nan
df["trans_prob"] = np.nan
df["log_trans_prob"] = np.nan
df["emiss_prob"] = np.nan
df["log_emiss_prob"] = np.nan
df["total_log_prob"] = np.nan

# Startwaarde voor begin
prev_log_prob = 0.0  # log(1)

for idx, row in df.iterrows():
    trans = row["transition"]

    if row["prev_state"] == "begin":
        trans_prob = 0.3
        log_trans_prob = np.log(trans_prob+ 1e-300)
        emiss_prob = 0.0  # geen onzekerheid bij begin
        log_emiss_prob = np.log(emiss_prob + 1e-300)

    elif trans in ["AIS→AIS", "AIS→RF", "RF→RF", "RF→AIS"]:
        prev = df.loc[idx - 1]
        
        try:
            if trans == "AIS→AIS":
                time = row["AIS_Timestamp"] - prev["AIS_Timestamp"]
                dist = haversine(row["AIS"], prev["AIS"], unit=Unit.KILOMETERS)
            elif trans == "AIS→RF":
                time = row["RF_Timestamp"] - prev["AIS_Timestamp"]
                dist = haversine(row["RF"], prev["AIS"], unit=Unit.KILOMETERS)
            elif trans == "RF→RF":
                time = row["RF_Timestamp"] - prev["RF_Timestamp"]
                dist = haversine(row["RF"], prev["RF"], unit=Unit.KILOMETERS)
            elif trans == "RF→AIS":
                time = row["AIS_Timestamp"] - prev["RF_Timestamp"]
                dist = haversine(row["AIS"], prev["RF"], unit=Unit.KILOMETERS)

            trans_prob = AIS_RF_probability(time.total_seconds(), dist)
            log_trans_prob = np.log(trans_prob + 1e-300)

        except Exception:
            trans_prob = 1e-10
            log_trans_prob = np.log(trans_prob)
        
        if row["AIS"] is not None and row["RF"] is None:
            emiss_prob = 1.0
        elif row["RF"] is not None and row["AIS"] is None:
            emiss_prob = 1.0
        else:
            emiss_prob = 0.0

        log_emiss_prob = np.log(emiss_prob + 1e-300)

    elif row["State"] == "end":
        trans_prob = 0.05
        log_trans_prob = np.log(trans_prob+ 1e-300)
        emiss_prob = 0.0
        log_emiss_prob = np.log(emiss_prob+ 1e-300)

    else:
        continue

    total_log_prob = prev_log_prob + log_trans_prob + log_emiss_prob

    # Schrijf naar DataFrame
    df.at[idx, "prev_log_prob"] = prev_log_prob
    df.at[idx, "trans_prob"] = trans_prob
    df.at[idx, "log_trans_prob"] = log_trans_prob
    df.at[idx, "emiss_prob"] = emiss_prob
    df.at[idx, "log_emiss_prob"] = log_emiss_prob
    df.at[idx, "total_log_prob"] = total_log_prob

    # Update voor volgende stap
    prev_log_prob = total_log_prob


In [16]:
df["trans_prob"].mean()

np.float64(0.9232637678630214)

In [17]:
df

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,prev_state,transition,trans_prob,prev_log_prob,log_trans_prob,emiss_prob,log_emiss_prob,total_log_prob
0,164,316009457,2,2024-01-03 00:01:08,NaT,NaT,None,None,begin,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,164,316009457,2,2024-01-03 00:01:09,2024-01-03 00:01:09,NaT,"[43.26082, -82.44656]",None,AIS,begin,begin→AIS,0.300000,0.000000,-1.203973,0.0,-690.775528,-691.979501
2,164,316009457,2,2024-01-03 00:02:59,2024-01-03 00:02:59,NaT,"[43.25506, -82.44536]",None,AIS,AIS,AIS→AIS,0.861682,-691.979501,-0.148869,1.0,0.000000,-692.128370
3,164,316009457,2,2024-01-03 00:04:09,2024-01-03 00:04:09,NaT,"[43.2514, -82.44465]",None,AIS,AIS,AIS→AIS,0.909681,-692.128370,-0.094661,1.0,0.000000,-692.223030
4,164,316009457,2,2024-01-03 00:05:20,2024-01-03 00:05:20,NaT,"[43.24775, -82.4439]",None,AIS,AIS,AIS→AIS,0.908807,-692.223030,-0.095623,1.0,0.000000,-692.318653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
917,164,316009457,2,2024-01-03 23:56:49,2024-01-03 23:56:49,NaT,"[42.26315, -83.10541]",None,AIS,AIS,AIS→AIS,0.933235,-786.467418,-0.069099,1.0,0.000000,-786.536517
918,164,316009457,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[42.26315, -83.10541]",None,AIS,AIS,AIS→AIS,0.931462,-786.536517,-0.071000,1.0,0.000000,-786.607517
919,164,316009457,2,2024-01-03 23:59:01,2024-01-03 23:59:01,NaT,"[42.26315, -83.1054]",None,AIS,AIS,AIS→AIS,0.940777,-786.607517,-0.061049,1.0,0.000000,-786.668566
920,164,316009457,2,2024-01-04 00:00:00,2024-01-04 00:00:00,NaT,"[42.26315, -83.1054]",None,AIS,AIS,AIS→AIS,0.942707,-786.668566,-0.059000,1.0,0.000000,-786.727566


RF: 164 en AIS: 291

In [18]:
df2 = test_set[test_set["TrackID"] == 291]
df2.head(5)
# sws rf verwijderen en match 

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
192668,291,366904890,2,2024-01-03 00:00:03,NaT,NaT,None,None,begin
192669,291,366904890,2,2024-01-03 00:00:04,2024-01-03 00:00:04,NaT,"[42.27507, -83.10986]",None,AIS
192670,291,366904890,2,2024-01-03 00:01:06,2024-01-03 00:01:06,NaT,"[42.27466, -83.10859]",None,AIS
192671,291,366904890,2,2024-01-03 00:02:54,2024-01-03 00:02:54,NaT,"[42.27421, -83.10593]",None,AIS
192672,291,366904890,2,2024-01-03 00:03:57,2024-01-03 00:03:57,NaT,"[42.27449, -83.10459]",None,AIS


In [19]:
df = test_set[test_set["TrackID"] == 164]
match_rows = df[df["State"] == "M"]

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df2 = test_set[test_set["TrackID"] == 291]

match_rows = df2[df2["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

df2 = df2[(df2["State"] != "M") & (df2["State"] != "RF")]
df2 = pd.concat([df2, ais_rows, rf_rows[0:1]], ignore_index=True)
df2 = df2.sort_values(by=["Timestamp"]).reset_index(drop=True) 
df2

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,291,366904890,2,2024-01-03 00:00:03,NaT,NaT,None,None,begin
1,291,366904890,2,2024-01-03 00:00:04,2024-01-03 00:00:04,NaT,"[42.27507, -83.10986]",None,AIS
2,291,366904890,2,2024-01-03 00:01:06,2024-01-03 00:01:06,NaT,"[42.27466, -83.10859]",None,AIS
3,291,366904890,2,2024-01-03 00:02:54,2024-01-03 00:02:54,NaT,"[42.27421, -83.10593]",None,AIS
4,291,366904890,2,2024-01-03 00:03:57,2024-01-03 00:03:57,NaT,"[42.27449, -83.10459]",None,AIS
...,...,...,...,...,...,...,...,...,...
1020,291,366904890,2,2024-01-03 23:55:41,2024-01-03 23:55:41,NaT,"[45.07417, -83.10874]",None,AIS
1021,291,366904890,2,2024-01-03 23:56:51,2024-01-03 23:56:51,NaT,"[45.0778, -83.10986]",None,AIS
1022,291,366904890,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[45.08135, -83.11099]",None,AIS
1023,291,366904890,2,2024-01-03 23:59:10,2024-01-03 23:59:10,NaT,"[45.08501, -83.11218]",None,AIS


In [20]:
df2 = df2[(df2["State"] != "M") & (df2["State"] != "RF")]
df2 = pd.concat([df2, rf_rows.iloc[0:1]], ignore_index=True)
df2 = df2.sort_values(by=["Timestamp"]).reset_index(drop=True) 
df2

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,291,366904890,2,2024-01-03 00:00:03,NaT,NaT,None,None,begin
1,291,366904890,2,2024-01-03 00:00:04,2024-01-03 00:00:04,NaT,"[42.27507, -83.10986]",None,AIS
2,291,366904890,2,2024-01-03 00:01:06,2024-01-03 00:01:06,NaT,"[42.27466, -83.10859]",None,AIS
3,291,366904890,2,2024-01-03 00:02:54,2024-01-03 00:02:54,NaT,"[42.27421, -83.10593]",None,AIS
4,291,366904890,2,2024-01-03 00:03:57,2024-01-03 00:03:57,NaT,"[42.27449, -83.10459]",None,AIS
...,...,...,...,...,...,...,...,...,...
1020,291,366904890,2,2024-01-03 23:55:41,2024-01-03 23:55:41,NaT,"[45.07417, -83.10874]",None,AIS
1021,291,366904890,2,2024-01-03 23:56:51,2024-01-03 23:56:51,NaT,"[45.0778, -83.10986]",None,AIS
1022,291,366904890,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[45.08135, -83.11099]",None,AIS
1023,291,366904890,2,2024-01-03 23:59:10,2024-01-03 23:59:10,NaT,"[45.08501, -83.11218]",None,AIS


In [21]:
df2["prev_state"] = df2["State"].shift(1)
df2["transition"] = df2["prev_state"] + "→" + df2["State"]
df2["trans_prob"] = np.nan

In [22]:
for id, row in df2.iterrows():
    trans = row["transition"]
    
    if row["prev_state"] == "begin":
        df2.at[id, "trans_prob"] = 0.3
    
    elif row["transition"] in ["AIS→AIS", "AIS→RF", "RF→RF", "RF→AIS"]:
        prev = df2.loc[id-1]
        if row["transition"] == "AIS→AIS":
            time = row["AIS_Timestamp"] - prev["AIS_Timestamp"]
            dist = haversine(row["AIS"], prev["AIS"], unit=Unit.KILOMETERS)
        elif row["transition"] == "AIS→RF":
            time = row["RF_Timestamp"] - prev["AIS_Timestamp"]
            dist = haversine(row["RF"], prev["AIS"], unit=Unit.KILOMETERS)
        elif row["transition"] == "RF→RF":
            time = row["RF_Timestamp"] - prev["RF_Timestamp"]
            dist = haversine(row["RF"], prev["RF"], unit=Unit.KILOMETERS)
        elif row["transition"] == "RF→AIS":
            time = row["AIS_Timestamp"] - prev["RF_Timestamp"]
            dist = haversine(row["AIS"], prev["RF"], unit=Unit.KILOMETERS)
        prob = AIS_RF_probability(time.total_seconds(), dist)
        log_trans_prob = np.log(prob + 1e-300)
        df2.at[id, "trans_prob"] = log_trans_prob
        
    elif row["State"] == "end":
        df2.at[id, "trans_prob"] = 0.05
        
df2

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,prev_state,transition,trans_prob
0,291,366904890,2,2024-01-03 00:00:03,NaT,NaT,None,None,begin,None,NaN,NaN
1,291,366904890,2,2024-01-03 00:00:04,2024-01-03 00:00:04,NaT,"[42.27507, -83.10986]",None,AIS,begin,begin→AIS,0.300000
2,291,366904890,2,2024-01-03 00:01:06,2024-01-03 00:01:06,NaT,"[42.27466, -83.10859]",None,AIS,AIS,AIS→AIS,-0.068840
3,291,366904890,2,2024-01-03 00:02:54,2024-01-03 00:02:54,NaT,"[42.27421, -83.10593]",None,AIS,AIS,AIS→AIS,-0.121470
4,291,366904890,2,2024-01-03 00:03:57,2024-01-03 00:03:57,NaT,"[42.27449, -83.10459]",None,AIS,AIS,AIS→AIS,-0.069874
...,...,...,...,...,...,...,...,...,...,...,...,...
1020,291,366904890,2,2024-01-03 23:55:41,2024-01-03 23:55:41,NaT,"[45.07417, -83.10874]",None,AIS,AIS,AIS→AIS,-0.082603
1021,291,366904890,2,2024-01-03 23:56:51,2024-01-03 23:56:51,NaT,"[45.0778, -83.10986]",None,AIS,AIS,AIS→AIS,-0.094786
1022,291,366904890,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[45.08135, -83.11099]",None,AIS,AIS,AIS→AIS,-0.093275
1023,291,366904890,2,2024-01-03 23:59:10,2024-01-03 23:59:10,NaT,"[45.08501, -83.11218]",None,AIS,AIS,AIS→AIS,-0.095054


In [23]:
df2["trans_prob"].mean()

np.float64(-0.11541469184184884)

In [ ]:
df2.iloc[52:56]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,prev_state,transition,trans_prob
52,291,366904890,2,2024-01-03 01:36:04,2024-01-03 01:36:04,NaT,"[42.3032, -83.08832]",None,AIS,AIS,AIS→AIS,0.955950
53,291,366904890,2,2024-01-03 01:39:03,2024-01-03 01:39:03,NaT,"[42.30321, -83.08833]",None,AIS,AIS,AIS→AIS,0.956217
54,164,316009457,2,2024-01-03 01:41:00,NaT,2024-01-03 01:41:00,None,"[42.966109086623185, -82.41862890079418]",RF,AIS,AIS→RF,0.244914
55,291,366904890,2,2024-01-03 01:42:05,2024-01-03 01:42:05,NaT,"[42.3032, -83.08831]",None,AIS,RF,RF→AIS,0.248119


In [ ]:
df2[df2]

In [ ]:
0.2 ** 2

0.04000000000000001

In [ ]:
0.2

0.2

In [ ]:
0.97 ** 2

0.9409

In [ ]:
AIS_RF_probability(10000, 400)

np.float64(0.00020346836901064417)

## 18:00

In [ ]:
begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
model = PHMM(states, end)

In [ ]:
test_set = pd.read_parquet("../data/processed/PHMM_sequence_data.parquet")
test_set

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,0,209112000,0,2024-01-03 13:19:57,NaT,NaT,None,None,begin
1,0,209112000,0,2024-01-03 13:19:58,2024-01-03 13:19:58,NaT,"[48.51548, -126.51201]",None,AIS
2,0,209112000,0,2024-01-03 13:32:50,2024-01-03 13:32:50,NaT,"[48.51033, -126.45776]",None,AIS
3,0,209112000,0,2024-01-03 13:34:50,2024-01-03 13:34:50,NaT,"[48.50968, -126.44914]",None,AIS
4,0,209112000,0,2024-01-03 13:40:59,2024-01-03 13:40:59,2024-01-03 13:41:00,"[48.50753, -126.42323]","[48.499443942360536, -126.4146031322306]",M
...,...,...,...,...,...,...,...,...,...
471750,710,677059800,0,2024-01-08 08:30:39,2024-01-08 08:30:39,NaT,"[19.00953, -68.1877]",None,AIS
471751,710,677059800,0,2024-01-08 08:47:23,2024-01-08 08:47:23,NaT,"[19.02076, -68.22079]",None,AIS
471752,710,677059800,0,2024-01-08 08:50:39,2024-01-08 08:50:39,NaT,"[19.02278, -68.22751]",None,AIS
471753,710,677059800,0,2024-01-08 09:41:00,NaT,2024-01-08 09:41:00,None,"[19.024754315587664, -68.23688644056789]",RF


In [ ]:
sequence = test_set[test_set['TrackID'] == 164]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None][0:1]

In [ ]:
RF_sequence

[(Timestamp('2024-01-03 01:41:00'), array([ 42.96610909, -82.4186289 ]))]

In [ ]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
df = test_set[test_set["TrackID"] == 164]
match_rows = df[df["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df = df[(df["State"] != "M") & (df["State"] != "RF")]
df = pd.concat([df, ais_rows, rf_rows.iloc[0:1]], ignore_index=True)
df = df.sort_values(by=["Timestamp"]).reset_index(drop=True) 

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged = df.copy().reset_index(drop=True)
df_merged["BestPathState"] = df_path["State"]
df_merged["LogProb"] = df_path["LogProb"]
df_merged["Probability"] = np.exp(df_path["LogProb"])

df_merged["prev_Timestamp"] = df_merged["Timestamp"].shift(1)
df_merged["delta_time_sec"] = (df_merged["Timestamp"] - df_merged["prev_Timestamp"]).dt.total_seconds()

df_merged["coord"] = df_merged["AIS"].combine_first(df_merged["RF"])
df_merged["prev_coord"] = df_merged["coord"].shift(1)
df_merged["distance_km"] = None  

for idx, row in df_merged.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged.at[idx, "distance_km"] = dist
df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km
0,164,316009457,2,2024-01-03 00:01:08,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None
1,164,316009457,2,2024-01-03 00:01:09,2024-01-03 00:01:09,NaT,"[43.26082, -82.44656]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:01:08,1.0,"[43.26082, -82.44656]",None,None
2,164,316009457,2,2024-01-03 00:02:59,2024-01-03 00:02:59,NaT,"[43.25506, -82.44536]",None,AIS,AIS,-1.282521,2.773373e-01,2024-01-03 00:01:09,110.0,"[43.25506, -82.44536]","[43.26082, -82.44656]",0.647814
3,164,316009457,2,2024-01-03 00:04:09,2024-01-03 00:04:09,NaT,"[43.2514, -82.44465]",None,AIS,AIS,-1.469395,2.300647e-01,2024-01-03 00:02:59,70.0,"[43.2514, -82.44465]","[43.25506, -82.44536]",0.411016
4,164,316009457,2,2024-01-03 00:05:20,2024-01-03 00:05:20,NaT,"[43.24775, -82.4439]",None,AIS,AIS,-1.659330,1.902665e-01,2024-01-03 00:04:09,71.0,"[43.24775, -82.4439]","[43.2514, -82.44465]",0.410383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
917,164,316009457,2,2024-01-03 23:56:49,2024-01-03 23:56:49,NaT,"[42.26315, -83.10541]",None,AIS,AIS,-78.873356,5.568482e-35,2024-01-03 23:55:40,69.0,"[42.26315, -83.10541]","[42.26315, -83.10539]",0.001646
918,164,316009457,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[42.26315, -83.10541]",None,AIS,AIS,-78.924649,5.290058e-35,2024-01-03 23:56:49,71.0,"[42.26315, -83.10541]","[42.26315, -83.10541]",0.0
919,164,316009457,2,2024-01-03 23:59:01,2024-01-03 23:59:01,NaT,"[42.26315, -83.1054]",None,AIS,AIS,-78.975942,5.025555e-35,2024-01-03 23:58:00,61.0,"[42.26315, -83.1054]","[42.26315, -83.10541]",0.000823
920,164,316009457,2,2024-01-04 00:00:00,2024-01-04 00:00:00,NaT,"[42.26315, -83.1054]",None,AIS,AIS,-79.027236,4.774277e-35,2024-01-03 23:59:01,59.0,"[42.26315, -83.1054]","[42.26315, -83.1054]",0.0


In [ ]:
df_merged["delta_time_sec"].mean(), df_merged["distance_km"].mean()

(np.float64(93.73832790445168), np.float64(0.15264899839240148))

In [ ]:
sequence = test_set[test_set['TrackID'] == 291]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

In [ ]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

In [ ]:
df = test_set[test_set["TrackID"] == 164]
match_rows = df[df["State"] == "M"]

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df2 = test_set[test_set["TrackID"] == 291]

match_rows = df2[df2["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

df2 = df2[(df2["State"] != "M") & (df2["State"] != "RF")]
df2 = pd.concat([df2, ais_rows, rf_rows[0:1]], ignore_index=True)
df2 = df2.sort_values(by=["Timestamp"]).reset_index(drop=True) 

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged = df2.copy().reset_index(drop=True)
df_merged["BestPathState"] = df_path["State"]
df_merged["LogProb"] = df_path["LogProb"]
df_merged["Probability"] = np.exp(df_path["LogProb"])

df_merged["prev_Timestamp"] = df_merged["Timestamp"].shift(1)
df_merged["delta_time_sec"] = (df_merged["Timestamp"] - df_merged["prev_Timestamp"]).dt.total_seconds()

df_merged["coord"] = df_merged["AIS"].combine_first(df_merged["RF"])
df_merged["prev_coord"] = df_merged["coord"].shift(1)
df_merged["distance_km"] = None  

for idx, row in df_merged.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged.at[idx, "distance_km"] = dist
df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km
0,291,366904890,2,2024-01-03 00:00:03,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None
1,291,366904890,2,2024-01-03 00:00:04,2024-01-03 00:00:04,NaT,"[42.27507, -83.10986]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:00:03,1.0,"[42.27507, -83.10986]",None,None
2,291,366904890,2,2024-01-03 00:01:06,2024-01-03 00:01:06,NaT,"[42.27466, -83.10859]",None,AIS,AIS,-1.202030,3.005834e-01,2024-01-03 00:00:04,62.0,"[42.27466, -83.10859]","[42.27507, -83.10986]",0.114003
3,291,366904890,2,2024-01-03 00:02:54,2024-01-03 00:02:54,NaT,"[42.27421, -83.10593]",None,AIS,AIS,-1.306942,2.706463e-01,2024-01-03 00:01:06,108.0,"[42.27421, -83.10593]","[42.27466, -83.10859]",0.224503
4,291,366904890,2,2024-01-03 00:03:57,2024-01-03 00:03:57,NaT,"[42.27449, -83.10459]",None,AIS,AIS,-1.412666,2.434932e-01,2024-01-03 00:02:54,63.0,"[42.27449, -83.10459]","[42.27421, -83.10593]",0.114563
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,291,366904890,2,2024-01-03 23:55:41,2024-01-03 23:55:41,NaT,"[45.07417, -83.10874]",None,AIS,AIS,-60.278245,6.629650e-27,2024-01-03 23:54:40,61.0,"[45.07417, -83.10874]","[45.07101, -83.10774]",0.360044
1021,291,366904890,2,2024-01-03 23:56:51,2024-01-03 23:56:51,NaT,"[45.0778, -83.10986]",None,AIS,AIS,-60.329539,6.298168e-27,2024-01-03 23:55:41,70.0,"[45.0778, -83.10986]","[45.07417, -83.10874]",0.413108
1022,291,366904890,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[45.08135, -83.11099]",None,AIS,AIS,-60.380832,5.983259e-27,2024-01-03 23:56:51,69.0,"[45.08135, -83.11099]","[45.0778, -83.10986]",0.404591
1023,291,366904890,2,2024-01-03 23:59:10,2024-01-03 23:59:10,NaT,"[45.08501, -83.11218]",None,AIS,AIS,-60.432125,5.684096e-27,2024-01-03 23:58:00,70.0,"[45.08501, -83.11218]","[45.08135, -83.11099]",0.417561


In [ ]:
df_merged["delta_time_sec"].mean(), df_merged["distance_km"].mean()

(np.float64(84.32421875), np.float64(0.5249289700921912))

In [ ]:
sequence3 = test_set[test_set['TrackID'] == 100]

AIS_sequence3 = [(time, coord) for time, coord in zip(sequence3["AIS_Timestamp"], sequence3["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence3 = [(time, coord) for time, coord in zip(sequence3["RF_Timestamp"], sequence3["RF"]) 
               if pd.notna(time) and coord is not None]

In [ ]:
path3, probability3, log_probs3 = model.viterbi(AIS_sequence3, RF_sequence3)

In [ ]:
RF_M_indexes_pred_path3 = [(i, state) for i, state in enumerate(path3) if state in {"RF", "M"}]
RF_M_indexes_real_path3 = [(i, state) for i, state in enumerate(sequence3["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path3}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path3}")

The indexes of RF and/or M of predicted path are: [(110, 'RF'), (286, 'RF')],       
The indexes of RF and/or M of the real path are: [(110, 'RF'), (286, 'RF')]


dit is gewoon even een test

In [ ]:
import math

In [ ]:
RF_data = test_set[test_set["RF"].notna()].copy()
RF_data["RF_data_point_id"] = RF_data.groupby("TrackID").cumcount() + 1

M_data = test_set[test_set["State"] == "M"].copy()
M_data["distance_km"] = M_data.apply(lambda row: haversine(row["AIS"], row["RF"], unit=Unit.KILOMETERS), axis=1)
radius_km = math.ceil(M_data["distance_km"].max())

viterbi_results = []

for _, RF_data_point in RF_data.iterrows():
    RF_coord = RF_data_point["RF"]
    RF_time = RF_data_point["RF_Timestamp"]
    
    for AIS_id, AIS_data in test_set.groupby("TrackID"):
        AIS_coords = [coord for coord in AIS_data["AIS"] if coord is not None]
        RF_within_radius = any(haversine(RF_coord, coord) <= radius_km for coord in AIS_coords)
        
        marge = pd.Timedelta(minutes=5)
        min_AIS_time = AIS_data["AIS_Timestamp"].min() - marge
        max_AIS_time = AIS_data["AIS_Timestamp"].max() + marge
        RF_within_time = min_AIS_time <= RF_time <= max_AIS_time
        
        if RF_within_radius and RF_within_time:
            AIS_seq = [(time, coord) 
                       for time, coord in zip(AIS_data["AIS_Timestamp"], AIS_data["AIS"]) 
                       if pd.notna(time) and coord is not None]
            RF_seq = [(RF_time, RF_coord)]
            
            path, prob = model.viterbi(AIS_seq, RF_seq)
            
            viterbi_results.append(
                {
                    "RF_sequence_id": RF_data_point["TrackID"],
                    "RF_data_point_id": RF_data_point["RF_data_point_id"],
                    "AIS_sequence_id": AIS_id,
                    "num_of_AIS_points": len(AIS_seq),
                    "viterbi_prob": prob,
                    "adjusted_viterbi_prob": prob/len(AIS_seq),
                    "match": RF_data_point["TrackID"] == AIS_id,
                    "predicted_path": path
                })
            

KeyboardInterrupt: 

In [ ]:
df_viterbi_results = pd.DataFrame(viterbi_results)
df_viterbi_results

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,adjusted_viterbi_prob,match,predicted_path
0,0,1,0,885,-52.568254,-0.059399,True,"[begin, AIS, AIS, AIS, AIS, RF, AIS, AIS, AIS,..."
1,0,1,436,805,-61.967455,-0.076978,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2,0,1,493,900,-82.319329,-0.091466,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
3,0,2,0,885,-86.539313,-0.097785,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
4,0,3,0,885,-82.780235,-0.093537,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
...,...,...,...,...,...,...,...,...
2354,682,3,682,658,-73.773259,-0.112117,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2355,683,1,181,770,-48.010525,-0.062351,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2356,683,1,682,658,-58.189184,-0.088433,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
2357,683,2,103,499,-36.664607,-0.073476,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [ ]:
df_viterbi_results[df_viterbi_results["RF_sequence_id"] == 164]

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,adjusted_viterbi_prob,match,predicted_path
555,164,1,164,919,-82.022968,-0.089252,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
556,164,1,291,1022,-63.427858,-0.062062,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
557,164,2,164,919,-97.086090,-0.105643,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
558,164,2,291,1022,-74.961829,-0.073348,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
559,164,3,164,919,-97.296356,-0.105872,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."
560,164,3,291,1022,-61.872262,-0.060540,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS..."


In [ ]:
AIS_RF_probability(400, 10)

np.float64(0.7788007830714049)